In [5]:
from pathlib import Path

import pandas as pd

from assignment4.dec_tree import id3, info_metrics
from assignment4.dec_tree.reduced_error_pruning import (
    train_validation_split,
    reduced_error_pruning,
    error_count,
)


In [6]:
# Supports launching Jupyter from the repo root or notebooks directory.
path = Path("data/networkTraffic.csv")
if not path.exists():
    path = Path("notebooks/data/networkTraffic.csv")

df = pd.read_csv(path)

target = "attack_cat"

categorical = [
    "proto", "service", "state", "is_sm_ips_ports",
]

numerical = [
    "dur", "spkts", "dpkts", "sbytes", "dbytes", "rate",
    "sttl", "dttl", "sload", "dload", "sloss", "dloss",
    "sinpkt", "dinpkt", "sjit", "djit", "swin", "stcpb",
    "dtcpb", "dwin", "tcprtt", "synack", "ackdat", "smean",
    "dmean", "trans_depth", "response_body_len", "ct_srv_src",
    "ct_state_ttl", "ct_dst_ltm", "ct_src_dport_ltm",
    "ct_dst_sport_ltm", "ct_dst_src_ltm", "ct_ftp_cmd",
    "ct_flw_http_mthd", "ct_src_ltm", "ct_srv_dst",
]

model_df = df[categorical + numerical + [target]].copy()

assert not model_df.isna().any().any(), "Missing values remain."
assert all(
    pd.api.types.is_numeric_dtype(model_df[col])
    for col in numerical
), "Some numerical columns were loaded as strings."

In [7]:
# A first run to check the entire training pipeline.
development_df = model_df.sample(
    n=min(1_000, len(model_df)),
    random_state=42,
)

training_df, validation_df = train_validation_split(
    development_df,
    validation_fraction=0.2,
    random_state=42,
)

print("Training rows:", len(training_df))
print("Validation rows:", len(validation_df))
print(training_df[target].value_counts())

Training rows: 800
Validation rows: 200
attack_cat
0    290
9    184
4    146
6     68
3     55
1     29
5     11
2     10
8      7
Name: count, dtype: int64


In [8]:
# Uses category values from training + validation, without using
# validation labels to grow the tree.
domains = {
    col: development_df[col].unique().tolist()
    for col in categorical
}

tree = id3.id3(
    df=training_df,
    info_metric=info_metrics.information_gains,
    domains=domains,
    continuous_features=numerical,
)

print("Tree induced.")

Tree induced.


In [9]:
def accuracy(tree, data):
    return 1 - error_count(tree, data) / len(data)

print(f"Training accuracy:   {accuracy(tree, training_df):.2%}")
print(f"Validation accuracy: {accuracy(tree, validation_df):.2%}")

Training accuracy:   99.62%
Validation accuracy: 76.00%


In [10]:
pruned_tree = reduced_error_pruning(
    tree,
    training_df,
    validation_df,
)

print(
    f"Pruned validation accuracy: "
    f"{accuracy(pruned_tree, validation_df):.2%}"
)

Pruned validation accuracy: 80.00%
